# Step 3 — Web Crawler

Goal of this notebook:

1. Fetch HTML from a URL.
2. Save HTML to `pages` table.
3. Record crawl results to `crawl_history` table.

> This notebook **does not** parse HTML or perform chunking. We only save raw HTML so it can be reprocessed anytime.

In [ ]:
from pathlib import Path
import sqlite3
import requests
from datetime import datetime
import time

DB_PATH = Path("../data/linux_docs.db")
assert DB_PATH.exists(), "Run Step 2 first."

USER_AGENT = "LinuxDocBot/1.0"
REQUEST_DELAY = 1

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

## URL to crawl

In [2]:
URL = "https://wiki.archlinux.org/title/Installation_guide"
URL


'https://wiki.archlinux.org/title/Installation_guide'

## Fetch HTML

In [ ]:
headers = {
    "User-Agent": USER_AGENT
}

time.sleep(REQUEST_DELAY)

response = requests.get(URL, headers=headers, timeout=30)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))

Status: 200
Content-Type: text/html; charset=UTF-8


## Save to SQLite

In [ ]:
html = response.text

title = None
if "<title>" in html:
    start = html.find("<title>") + len("<title>")
    end = html.find("</title>")
    title = html[start:end].strip()

cursor.execute(
    '''
    INSERT INTO pages(url, title, html, status, scraped_at)
    VALUES (?, ?, ?, ?, ?)
    ON CONFLICT(url)
    DO UPDATE SET
        title=excluded.title,
        html=excluded.html,
        status=excluded.status,
        scraped_at=excluded.scraped_at
    ''',
    (
        URL,
        title,
        html,
        str(response.status_code),
        datetime.utcnow().isoformat()
    )
)

cursor.execute(
    '''
    INSERT INTO crawl_history(url, last_crawled, status_code, etag, last_modified)
    VALUES (?, ?, ?, ?, ?)
    ON CONFLICT(url)
    DO UPDATE SET
        last_crawled=excluded.last_crawled,
        status_code=excluded.status_code,
        etag=excluded.etag,
        last_modified=excluded.last_modified
    ''',
    (
        URL,
        datetime.utcnow().isoformat(),
        response.status_code,
        response.headers.get("ETag"),
        response.headers.get("Last-Modified")
    )
)

conn.commit()

print("✅ Page saved successfully.")

✅ Halaman berhasil disimpan.


## Verification

In [ ]:
cursor.execute("""
SELECT id, url, title, scraped_at
FROM pages
ORDER BY id DESC
LIMIT 5;
""")

for row in cursor.fetchall():
    print(dict(row))

{'id': 1, 'url': 'https://wiki.archlinux.org/title/Installation_guide', 'title': 'Installation guide - ArchWiki', 'scraped_at': '2026-07-17T07:38:06.246065'}


In [ ]:
conn.close()
print("Done ✅")

Selesai 🎉
